In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
import shap

CONTRACT_FEATURES = [
    "amount_inr",
    "active_call",
    "call_duration_seconds",
    "screen_share_active",
    "typing_jitter_ms",
    "is_new_payee",
    "velocity_10m",
]


def generate_contract_dataset(n_samples=25000, fraud_ratio=0.06):
  np.random.seed(42)
  n_fraud = int(n_samples * fraud_ratio)
  n_legit = n_samples - n_fraud

  # Legitimate user baseline distributions (with realistic overlap noise)
  legit_df = pd.DataFrame({
      "amount_inr": np.random.exponential(scale=2200, size=n_legit).clip(
          10.0, 95000.0
      ),
      "active_call": np.random.binomial(1, 0.05, size=n_legit),
      "call_duration_seconds": np.random.exponential(
          scale=180, size=n_legit
      ).clip(0, 7200),
      "screen_share_active": np.random.binomial(1, 0.008, size=n_legit),
      "typing_jitter_ms": np.random.normal(loc=55.0, scale=22.0, size=n_legit).clip(
          10.0, 500.0
      ),
      "is_new_payee": np.random.binomial(1, 0.18, size=n_legit),
      "velocity_10m": np.random.poisson(lam=1.1, size=n_legit).clip(1, 20),
      "is_fraud": 0,
  })

  # Coercion / APP Fraud / Mule distributions
  fraud_df = pd.DataFrame({
      "amount_inr": np.random.uniform(8500.0, 98000.0, size=n_fraud),
      "active_call": np.random.binomial(1, 0.78, size=n_fraud),
      "call_duration_seconds": np.random.normal(
          loc=1600, scale=450, size=n_fraud
      ).clip(0, 7200),
      "screen_share_active": np.random.binomial(1, 0.42, size=n_fraud),
      "typing_jitter_ms": np.random.normal(
          loc=155.0, scale=45.0, size=n_fraud
      ).clip(10.0, 500.0),
      "is_new_payee": np.random.binomial(1, 0.85, size=n_fraud),
      "velocity_10m": np.random.choice(
          [1, 2, 3, 5, 8], size=n_fraud, p=[0.40, 0.30, 0.15, 0.10, 0.05]
      ),
      "is_fraud": 1,
  })

  # Introduce 3% label flip / borderline transactions to prevent artificial 1.0 metrics
  df = (
      pd.concat([legit_df, fraud_df])
      .sample(frac=1.0, random_state=42)
      .reset_index(drop=True)
  )
  noise_mask = np.random.rand(len(df)) < 0.015
  df.loc[noise_mask, "is_fraud"] = 1 - df.loc[noise_mask, "is_fraud"]

  # Type alignments
  df["active_call"] = df["active_call"].astype(int)
  df["call_duration_seconds"] = df["call_duration_seconds"].astype(int)
  df["screen_share_active"] = df["screen_share_active"].astype(int)
  df["is_new_payee"] = df["is_new_payee"].astype(int)
  df["velocity_10m"] = df["velocity_10m"].astype(int)

  os.makedirs("ml/data", exist_ok=True)
  df.to_csv("ml/data/transactions.csv", index=False)
  return df


def train_pipeline():
  df = generate_contract_dataset()
  X = df[CONTRACT_FEATURES]
  y = df["is_fraud"]

  # 70% Train, 15% Calibration, 15% Test
  idx_train = int(len(df) * 0.70)
  idx_calib = int(len(df) * 0.85)

  X_train, y_train = X.iloc[:idx_train], y.iloc[:idx_train]
  X_calib, y_calib = X.iloc[idx_train:idx_calib], y.iloc[idx_train:idx_calib]
  X_test, y_test = X.iloc[idx_calib:], y.iloc[idx_calib:]

  scale_pos = (len(y_train) - sum(y_train)) / max(1, sum(y_train))

  base_model = LGBMClassifier(
    n_estimators=250,       # Increased from 120 for better learning
    learning_rate=0.02,     # Lowered to learn finer patterns
    max_depth=7,            # Deepened from 5
    num_leaves=45,          # Increased from 28
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=scale_pos,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
  base_model.fit(X_train, y_train)

  # Sigmoid (Platt) Calibration
  calibrator = CalibratedClassifierCV(
      estimator=base_model, method="sigmoid", cv="prefit"
  )
  calibrator.fit(X_calib, y_calib)

  # TreeSHAP Explainer
  explainer = shap.TreeExplainer(base_model.booster_)

  # Evaluation
  test_probs = calibrator.predict_proba(X_test)[:, 1]
  print("=" * 60)
  print("REVISED CONTRACT MODEL EVALUATION")
  print("=" * 60)
  print(f"PR-AUC:   {average_precision_score(y_test, test_probs):.4f}")
  print(f"ROC-AUC:  {roc_auc_score(y_test, test_probs):.4f}")
  print(f"Brier:    {brier_score_loss(y_test, test_probs):.4f}")

  # Single-file bundle required by Contract Section 5.2
  model_bundle = {
      "calibrator": calibrator,
      "base_model": base_model,
      "explainer": explainer,
      "features": CONTRACT_FEATURES,
  }

  os.makedirs("ml/saved_models", exist_ok=True)
  export_target = "ml/saved_models/fraud_model.joblib"
  joblib.dump(model_bundle, export_target)
  print(f"\nModel artifact bundled and saved to: {export_target}")


if __name__ == "__main__":
  train_pipeline()

/usr/local/lib/python3.13/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


REVISED CONTRACT MODEL EVALUATION
PR-AUC:   0.8583
ROC-AUC:  0.9114
Brier:    0.0155

Model artifact bundled and saved to: ml/saved_models/fraud_model.joblib
